# Lesson 6: The mathematical trick in self-attention

Self-attention is a weighted sum: each position gathers information from positions it is allowed to see. We will build that operation four ways, starting with ordinary loops and ending with query-key scores. To focus on the operation, this lesson uses one tiny sequence (batch size 1), with four time steps and two channels.

The matching [C entry point](../c/src/attention_math_main.c) and [Java lesson](../java/src/main/java/learning/karpathy/gpt1/SelfAttentionMathLesson.java) use the same four small input vectors and fixed projection matrices. Their arrays make the intermediate calculations visible without a tensor library.

## A tiny sequence of vectors

In [ ]:
import torch
import torch.nn.functional as F

# Each row is one time step; each vector has two channels.
x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 2.0],
        [3.0, 1.0],
        [2.0, 4.0],
    ],
    dtype=torch.float64,
)
time_steps, channels = x.shape

print(f"Input shape: [time={time_steps}, channels={channels}]")
print(x)

## Step 1: Average the current vector with its past

At time t, include the vectors from time 0 through t and average them. This is causal: a position never reads a future vector. The loop makes that rule explicit.

In [ ]:
loop_output = torch.zeros_like(x)

for time in range(time_steps):
    visible_past = x[: time + 1]
    loop_output[time] = visible_past.mean(dim=0)

print("Causal averages computed with loops:")
print(loop_output)

## Step 2: Express the averages as a matrix multiply

A lower-triangular matrix represents which time steps can contribute. Divide each row by its number of visible positions, then multiply the weights by the input vectors. This gives the same averages as the loops.

In [ ]:
causal_weights = torch.tril(torch.ones(time_steps, time_steps, dtype=x.dtype))
causal_weights = causal_weights / causal_weights.sum(dim=1, keepdim=True)

matrix_output = causal_weights @ x
print("Lower-triangular averaging weights:")
print(causal_weights)
print("Weights multiplied by input vectors:")
print(matrix_output)

torch.testing.assert_close(matrix_output, loop_output)

## Step 3: Build the causal weights with a mask and softmax

Instead of pre-normalizing the lower triangle, start with equal scores. Replace future scores with negative infinity. Softmax turns those masked positions into exact zeros and normalizes each remaining row. Equal visible scores produce the same uniform causal averages.

In [ ]:
future_mask = torch.triu(
    torch.ones(time_steps, time_steps, dtype=torch.bool),
    diagonal=1,
)
zero_scores = torch.zeros(time_steps, time_steps, dtype=x.dtype)
masked_scores = zero_scores.masked_fill(future_mask, float("-inf"))
softmax_weights = F.softmax(masked_scores, dim=-1)
softmax_output = softmax_weights @ x

print("Scores after masking future positions:")
print(masked_scores)
print("Softmax weights:")
print(softmax_weights)
print("Weighted vectors:")
print(softmax_output)

torch.testing.assert_close(softmax_output, loop_output)
assert torch.all(softmax_weights[future_mask] == 0)

## Step 4: Let query-key similarity choose the weights

Now each token has a query and a key. A query represents what the current position is looking for; a key represents what each candidate position offers. Their dot product becomes an attention score.

The fixed projection matrices below stand in for learned linear layers. We scale scores by the square root of the channel count, apply the same causal mask, and use the original input vectors as values. In a full attention head, a learned value projection supplies those value vectors too.

In [ ]:
query_projection = torch.tensor(
    [[1.0, 0.5], [-0.5, 1.0]],
    dtype=x.dtype,
)
key_projection = torch.tensor(
    [[0.5, -0.5], [1.0, 0.5]],
    dtype=x.dtype,
)

queries = x @ query_projection
keys = x @ key_projection
values = x

# Each query is compared with every key, then scaled to keep scores controlled.
attention_scores = (queries @ keys.T) / (channels ** 0.5)
masked_attention_scores = attention_scores.masked_fill(future_mask, float("-inf"))
attention_weights = F.softmax(masked_attention_scores, dim=-1)
attention_output = attention_weights @ values

print("Queries:")
print(queries)
print("Keys:")
print(keys)
print("Scaled query-key scores after the causal mask:")
print(masked_attention_scores)
print("Input-dependent attention weights:")
print(attention_weights)
print("Weighted value vectors:")
print(attention_output)

torch.testing.assert_close(attention_weights.sum(dim=-1), torch.ones(time_steps, dtype=x.dtype))
assert torch.all(attention_weights[future_mask] == 0)

## Compare and experiment

Run **Terminal → Run Task → Run tutorial lesson**, choose lesson **6**, then select a step or range. The C and Java outputs show the same input, projections, causal masks, and weights as this notebook.

Try changing the input vectors or projection matrices. Which positions receive more attention? What changes if you remove the causal mask? Why does softmax assign exactly zero weight to a masked position? The fixed projections are examples only; training a model would learn those matrices from data.

References: [PyTorch lower-triangular matrices](https://docs.pytorch.org/docs/stable/generated/torch.tril.html), [softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html), [matrix multiplication](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html), and the [original GPT tutorial notebook](../gpt_dev.ipynb), cells 18–26.